# HMS – Harmful Brain Activity Classification: a simple scikit-learn baseline

**Task.** Each sample is an EEG segment from a critically ill patient. Several experts voted on which of six
patterns it shows: seizure, LPD, GPD, LRDA, GRDA or other. The target is the *distribution* of votes,
and submissions are scored with the KL divergence between the true and the predicted distribution (lower is better).

**Approach.**
1. **Labels**: for each EEG, turn the expert votes into a probability distribution over the six classes.
2. **Features** (88 per EEG): from the official Kaggle spectrograms we take the 10 minutes around the labelled
   segment and compute log band power (delta, theta, alpha, beta) for each of the four scalp regions, summarised
   over time (mean, std, min, max), plus relative band power and left–right asymmetry.
3. **Models**: `LogisticRegression` and `HistGradientBoostingClassifier` from scikit-learn.
   Soft labels are handled by repeating each EEG once per class with `sample_weight` = vote fraction;
   the weighted cross-entropy is then equal to the KL divergence up to a constant.
4. **Validation**: 5-fold `GroupKFold` grouped by `patient_id`, so no patient appears in both the training and
   the validation part of a fold. The model with the better cross-validation (CV) score is used for the submission.
5. **Submission**: refit the chosen model on all training data and predict the hidden test set.

**Settings** (next cell): `MODE = "full"` uses all training spectrograms (about 10 minutes on a Kaggle CPU);
`MODE = "quick"` uses a random 3,000 for a fast check. `RUN_CV = False` skips cross-validation.

Runs on CPU only; internet must be off for Kaggle submissions.

In [1]:
print("Hello world")
import os
import time
import warnings
from glob import glob
from mne.filter import filter_data, notch_filter

import numpy as np
import pandas as pd
from joblib import Parallel, delayed
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# Some spectrograms contain frequency bands that are entirely NaN; numpy warns about the
# resulting empty means. These cases are handled explicitly below, so the warnings are muted.
warnings.filterwarnings("ignore", category=RuntimeWarning)

# ----------------------------------------------------------------------------- settings
MODE = "quick"      # "full": all training spectrograms | "quick": a random subset for a fast check
RUN_CV = False      # True: run 5-fold cross-validation, report CV scores and pick the better model
MAX_SPECS = {"full": None, "quick": 3000}[MODE]   # number of training spectrograms to read (None = all)
N_FOLDS = 5        # number of cross-validation folds
N_JOBS = -1        # number of CPU cores used to read spectrograms in parallel (-1 = all cores)
SEED = 42          # random seed for subsampling and for the gradient-boosting model
FEATURE_SET = "spec"    # "spec" = alleen spectrogram | "eeg" = alleen ruwe EEG | "both" = beide
# ----------------------------------------------------------------------------- data location
# Kaggle mounts the competition data at different paths depending on the notebook environment.
# HMS_DATA can be set to run the notebook elsewhere (e.g. on a local copy of the data).
CANDIDATES = ["/kaggle/input/competitions/hms-harmful-brain-activity-classification",
              "/kaggle/input/hms-harmful-brain-activity-classification",
              r"C:\MachineLearning opdracht"]                                                                    #nu kan ik runnen op kaggle en lokaal
DATA = next((p for p in CANDIDATES if os.path.exists(os.path.join(p, "train.csv"))), None)
if DATA is None:
    raise FileNotFoundError("train.csv niet gevonden in een van de paden in CANDIDATES.")
print("DATA =", DATA)

# ----------------------------------------------------------------------------- constants
# Vote columns in train.csv, in the order required by the submission file.
CLASSES = ["seizure_vote", "lpd_vote", "gpd_vote", "lrda_vote", "grda_vote", "other_vote"]
# The official spectrograms average the bipolar EEG channels into four scalp regions:
# LL = left lateral, RL = right lateral, LP = left parasagittal, RP = right parasagittal.
REGIONS = ["LL", "RL", "LP", "RP"]
# Classical EEG frequency bands in Hz (the spectrograms cover roughly 0.5-20 Hz).
BANDS = {"delta": (0.5, 4), "theta": (4, 8), "alpha": (8, 13), "beta": (13, 20)}

Hello world
DATA = /kaggle/input/competitions/hms-harmful-brain-activity-classification


## 1. Labels: one row per EEG, votes normalised to a probability distribution

`train.csv` has several rows (labelled sub-segments) per `eeg_id`. We sum the votes over these rows,
divide by the total number of votes, and keep the range of label offsets to locate the window in the spectrogram.

In [2]:
train = pd.read_csv(f"{DATA}/train.csv")

# One row per EEG: spectrogram id, first/last label offset (seconds), patient id and summed votes per class.
agg = {"spectrogram_id": "first", "spectrogram_label_offset_seconds": ["min", "max"],
       "eeg_label_offset_seconds": ["min", "max"], "patient_id": "first"}
agg.update({c: "sum" for c in CLASSES})
df = train.groupby("eeg_id").agg(agg)
df.columns = ["spectrogram_id", "off_min", "off_max", "eeg_off_min", "eeg_off_max", "patient_id"] + CLASSES

# Convert vote counts into a probability distribution (each row sums to 1).
df["num_votes"] = df[CLASSES].sum(axis=1)
df[CLASSES] = df[CLASSES].values / df["num_votes"].values[:, None]
df = df.reset_index()
# Het 50 s-fragment begint op de offset; het label hoort bij de centrale 10 s (dus +25 s).
df["eeg_center"] = (df["eeg_off_min"] + df["eeg_off_max"]) / 2 + 25

# Quick mode: keep only the EEGs of a random subset of spectrograms.
if MAX_SPECS:
    keep = pd.Series(df.spectrogram_id.unique()).sample(n=min(MAX_SPECS, df.spectrogram_id.nunique()),
                                                         random_state=SEED)
    df = df[df.spectrogram_id.isin(keep)].reset_index(drop=True)

print(df.shape, "EEGs from", df.patient_id.nunique(), "patients,", df.spectrogram_id.nunique(), "spectrograms")
df.head()

(4738, 15) EEGs from 1163 patients, 3000 spectrograms


,eeg_id,spectrogram_id,off_min,off_max,eeg_off_min,eeg_off_max,patient_id,seizure_vote,lpd_vote,gpd_vote,lrda_vote,grda_vote,other_vote,num_votes,eeg_center
0,568657,789577333,0.0,16.0,0.0,16.0,20654,0.0,0.0,0.250000,0.000000,0.166667,0.583333,48,33.0
1,751790,618728447,908.0,908.0,0.0,0.0,38549,0.0,0.0,1.000000,0.000000,0.000000,0.000000,1,25.0
2,2078097,2074135650,3342.0,3342.0,0.0,0.0,61174,0.0,0.0,0.000000,0.000000,0.000000,1.000000,2,25.0
3,2482631,978166025,1902.0,1944.0,0.0,42.0,20606,0.0,0.0,0.133333,0.066667,0.133333,0.666667,105,46.0
4,2918824,1211648246,3462.0,3462.0,0.0,0.0,14965,1.0,0.0,0.000000,0.000000,0.000000,0.000000,2,25.0


## 2. Features from the official spectrograms

Each spectrogram file has one row every 2 seconds and 400 power columns named `<region>_<frequency>`
(e.g. `LL_1.17`). We take a 10-minute window (300 rows) and reduce it to 88 numbers per EEG:

| Feature group | Count |
|---|---|
| Log band power per region and band: mean, std, min, max over time | 4 regions × 4 bands × 4 = 64 |
| Relative band power (band power / total power of the region) | 4 × 4 = 16 |
| Left–right asymmetry of mean log band power (lateral and parasagittal chains) | 2 × 4 = 8 |

Spectrogram files are read in parallel because reading them is the slowest step of the notebook.

In [3]:
def band_columns(columns):
    """Return {(region, band): [column indices]} for the 400 spectrogram columns like 'LL_1.17'."""
    out = {}
    for r in REGIONS:
        for b, (lo, hi) in BANDS.items():
            out[(r, b)] = [i for i, c in enumerate(columns)
                           if c.startswith(r + "_") and lo <= float(c.split("_")[1]) < hi]
    return out


def window_features(spec, cols):
    """Compute the 88 features for one 10-minute window.

    spec: float array of shape (time steps, 400 power columns).
    cols: output of band_columns().
    """
    feats = {}
    total = {r: 0.0 for r in REGIONS}   # total power per region, for relative band power
    band_power = {}
    for (r, b), idx in cols.items():
        p = np.nanmean(spec[:, idx], axis=1)                     # mean power in the band, per time step
        lp = np.log1p(np.nan_to_num(p, nan=0.0).clip(min=0))     # log scale; missing values -> 0
        feats[f"{r}_{b}_mean"] = lp.mean()
        feats[f"{r}_{b}_std"] = lp.std()
        feats[f"{r}_{b}_min"] = lp.min()
        feats[f"{r}_{b}_max"] = lp.max()
        band_power[(r, b)] = np.nanmean(p) if np.isfinite(p).any() else 0.0
        total[r] += band_power[(r, b)]
    for (r, b), v in band_power.items():                         # share of the region's total power
        feats[f"{r}_{b}_rel"] = v / (total[r] + 1e-6)
    for b in BANDS:                                              # left minus right hemisphere
        feats[f"asym_lat_{b}"] = feats[f"LL_{b}_mean"] - feats[f"RL_{b}_mean"]
        feats[f"asym_par_{b}"] = feats[f"LP_{b}_mean"] - feats[f"RP_{b}_mean"]
    return feats



In [4]:
# halve breedte in rijen (1 rij = 2 s): c10 = 10 s, c50s = 50 s, c60 = 1 min
SCALES = {"c10": 2, "c50s": 12, "c60": 15, "full": None}


def mean_feats(sub, cols):
    """Alleen de 16 band-gemiddelden (regio x band), voor de randvensters."""
    if len(sub) == 0:
        return None
    f = window_features(sub, cols)
    return {k: v for k, v in f.items() if k.endswith("_mean") and k.split("_")[0] in REGIONS}


def multiscale_features(seg, cols):
    mid = len(seg) // 2
    feats = {}

    # 1. Lagen rond het midden
    for name, half in SCALES.items():
        sub = seg if half is None else seg[max(0, mid - half): mid + half + 1]
        if len(sub) == 0:
            sub = seg
        for k, v in window_features(sub, cols).items():
            feats[f"{name}_{k}"] = v

    # 2. Centrum minus omgeving, voor elke laag groter dan c10
    for name in SCALES:
        if name == "c10":
            continue
        for r in REGIONS:
            for b in BANDS:
                feats[f"diff_c10_{name}_{r}_{b}"] = feats[f"c10_{r}_{b}_mean"] - feats[f"{name}_{r}_{b}_mean"]

    # 3. Randen van het 50 s-fragment: ongeveer 12-24 s voor en na het midden
    left = mean_feats(seg[max(0, mid - 12): max(0, mid - 5)], cols)
    right = mean_feats(seg[mid + 6: mid + 13], cols)
    for r in REGIONS:
        for b in BANDS:
            key = f"{r}_{b}_mean"
            l = left[key] if left else feats[f"full_{key}"]
            rt = right[key] if right else feats[f"full_{key}"]
            feats[f"edgeL_{r}_{b}"] = l
            feats[f"edgeR_{r}_{b}"] = rt
            feats[f"edge_vs_c10_{r}_{b}"] = (l + rt) / 2 - feats[f"c10_{key}"]
    return feats


In [5]:



def spectrogram_features(path, windows, is_test):
    """Read one spectrogram file and compute features for every EEG that uses it.

    windows: list of (eeg_id, off_min, off_max); offsets are the label positions in seconds.
    """
    spec = pd.read_parquet(path)
    cols = band_columns(list(spec.columns[1:]))
    spec = spec.iloc[:, 1:].to_numpy(dtype=np.float32)          # drop the 'time' column
    rows = []
    for eeg_id, off_min, off_max in windows:
        if is_test:
            # Test spectrograms are short and centred on the segment: take their middle 10 minutes.
            start = max(0, (len(spec) - 300) // 2)
        else:
            # Training spectrograms cover the whole recording. Start at the midpoint of the labelled
            # offsets; offsets are in seconds and rows are 2 s apart, hence the division by 4 (= /2 /2).
            start = int((off_min + off_max) // 4)
        feats = multiscale_features(spec[start:start + 300], cols)
        feats["eeg_id"] = eeg_id
        rows.append(feats)
    return rows


def build_features(meta, spec_dir, is_test=False):
    """Compute features for all EEGs in `meta`, reading each spectrogram file once (in parallel)."""
    t0 = time.time()
    if is_test:   # test.csv has no label offsets
        meta = meta.assign(off_min=0.0, off_max=0.0)
    jobs = [delayed(spectrogram_features)(f"{spec_dir}/{sid}.parquet",
                                          list(zip(g.eeg_id, g.off_min, g.off_max)), is_test)
            for sid, g in meta.groupby("spectrogram_id")]
    results = Parallel(n_jobs=N_JOBS, batch_size=32)(jobs)
    out = pd.DataFrame([row for rows in results for row in rows])
    print(f"{len(jobs)} spectrograms -> {len(out)} feature rows in {time.time() - t0:.0f}s")
    return out


X_df = build_features(df, f"{DATA}/train_spectrograms")
df = df.merge(X_df, on="eeg_id")
FEATS = [c for c in X_df.columns if c != "eeg_id"]
print(len(FEATS), "features")

/tmp/ipykernel_50/1471651478.py:21: RuntimeWarning: Mean of empty slice
/tmp/ipykernel_50/1471651478.py:21: RuntimeWarning: Mean of empty slice
/tmp/ipykernel_50/1471651478.py:21: RuntimeWarning: Mean of empty slice
/tmp/ipykernel_50/1471651478.py:21: RuntimeWarning: Mean of empty slice
/tmp/ipykernel_50/1471651478.py:21: RuntimeWarning: Mean of empty slice
/tmp/ipykernel_50/1471651478.py:21: RuntimeWarning: Mean of empty slice
/tmp/ipykernel_50/1471651478.py:21: RuntimeWarning: Mean of empty slice
/tmp/ipykernel_50/1471651478.py:21: RuntimeWarning: Mean of empty slice
/tmp/ipykernel_50/1471651478.py:21: RuntimeWarning: Mean of empty slice
/tmp/ipykernel_50/1471651478.py:21: RuntimeWarning: Mean of empty slice
/tmp/ipykernel_50/1471651478.py:21: RuntimeWarning: Mean of empty slice
/tmp/ipykernel_50/1471651478.py:21: RuntimeWarning: Mean of empty slice
/tmp/ipykernel_50/1471651478.py:21: RuntimeWarning: Mean of empty slice
/tmp/ipykernel_50/1471651478.py:21: RuntimeWarning: Mean of empt

3000 spectrograms -> 4738 feature rows in 76s
448 features


In [6]:
from scipy.signal import welch

FS = 200   # EEG: 200 samples per seconde

# Deotte's formule: per regio vier bipolaire paren. Per paar een spectrum, daarna het GEMIDDELDE
# van die vier spectra. Niet eerst de signalen optellen, want dan vallen de tussenliggende elektroden weg.
MONTAGE = {
    "LL": [("Fp1", "F7"), ("F7", "T3"), ("T3", "T5"), ("T5", "O1")],
    "LP": [("Fp1", "F3"), ("F3", "C3"), ("C3", "P3"), ("P3", "O1")],
    "RP": [("Fp2", "F4"), ("F4", "C4"), ("C4", "P4"), ("P4", "O2")],
    "RL": [("Fp2", "F8"), ("F8", "T4"), ("T4", "T6"), ("T6", "O2")],
}
PAIRS = [(a, b) for pairs in MONTAGE.values() for a, b in pairs]      # 16 paren, 4 per regio, in MONTAGE-volgorde
EEG_CHANNELS = sorted({c for a, b in PAIRS for c in (a, b)})
CH_IDX = {c: i for i, c in enumerate(EEG_CHANNELS)}


def eeg_window_features(x, prefix):
    """x: (samples, kanalen in volgorde van EEG_CHANNELS). Geeft de features voor één tijdvenster."""
    bip = np.stack([x[:, CH_IDX[a]] - x[:, CH_IDX[b]] for a, b in PAIRS], axis=1)       # (samples, 16)
    f, pxx = welch(bip, fs=FS, nperseg=min(400, len(bip)), axis=0)                        # 2 s-segmenten, 0.5 Hz
    feats, logpow = {}, {}
    for i, r in enumerate(MONTAGE):
        spec = pxx[:, 4 * i: 4 * i + 4].mean(axis=1)                                      # gemiddelde van de 4 spectra
        total = spec[(f >= 0.5) & (f < 20)].sum() + 1e-12
        for b, (lo, hi) in BANDS.items():
            m = (f >= lo) & (f < hi)
            logpow[(r, b)] = np.log10(spec[m].mean() + 1e-6)
            feats[f"{prefix}_{r}_{b}_logpow"] = logpow[(r, b)]
            feats[f"{prefix}_{r}_{b}_relpow"] = spec[m].sum() / total
        sig = bip[:, 4 * i: 4 * i + 4]
        feats[f"{prefix}_{r}_linelen"] = np.abs(np.diff(sig, axis=0)).mean()              # scherpte/pieken
        feats[f"{prefix}_{r}_std"] = sig.std()
    for b in BANDS:                                                                        # links minus rechts
        feats[f"{prefix}_asym_lat_{b}"] = logpow[("LL", b)] - logpow[("RL", b)]
        feats[f"{prefix}_asym_par_{b}"] = logpow[("LP", b)] - logpow[("RP", b)]
    return feats


def eeg_features(path, center_sec, eeg_id, is_test):
    x = pd.read_parquet(path, columns=EEG_CHANNELS).to_numpy(dtype=np.float32)
    x = np.nan_to_num(x, nan=0.0)
    n50 = 50 * FS
    start = 0 if is_test else int(center_sec * FS) - n50 // 2        # test_eegs zijn precies 50 s
    start = int(np.clip(start, 0, max(0, len(x) - n50)))
    seg = x[start:start + n50]
    
    if len(seg) < n50:                                                # vangnet: te korte opname
        seg = np.pad(seg, ((0, n50 - len(seg)), (0, 0)))
    seg = notch_filter(seg.T.astype("float64"), FS, 60, verbose="ERROR")        # 60 Hz storing
    seg = filter_data(seg, FS, 0.5, 40, verbose="ERROR").T.astype("float32")     # 0,5-40 Hz
    mid = len(seg) // 2
    feats = eeg_window_features(seg, "eeg50")                                         # hele 50 s
    feats.update(eeg_window_features(seg[mid - 5 * FS: mid + 5 * FS], "eeg10"))       # centrale 10 s
    for k in [k for k in feats if k.startswith("eeg10_") and k.endswith("logpow")]:   # centrum minus context
        feats["eegdiff_" + k[len("eeg10_"):]] = feats[k] - feats[k.replace("eeg10_", "eeg50_")]
    feats["eeg_id"] = eeg_id
    return feats


def build_eeg_features(meta, eeg_dir, is_test=False):
    t0 = time.time()
    centers = [25.0] * len(meta) if is_test else meta["eeg_center"].tolist()
    jobs = [delayed(eeg_features)(f"{eeg_dir}/{e}.parquet", c, e, is_test)
            for e, c in zip(meta["eeg_id"], centers)]
    out = pd.DataFrame(Parallel(n_jobs=N_JOBS, batch_size=16)(jobs))
    print(f"{len(jobs)} EEG's -> {out.shape[1] - 1} features in {time.time() - t0:.0f}s")
    return out


SPEC_FEATS = list(FEATS)
if FEATURE_SET != "spec":
    E_df = build_eeg_features(df, f"{DATA}/train_eegs")
    df = df.merge(E_df, on="eeg_id")
    EEG_FEATS = [c for c in E_df.columns if c != "eeg_id"]
else:
    EEG_FEATS = []
FEATS = {"spec": SPEC_FEATS, "eeg": EEG_FEATS, "both": SPEC_FEATS + EEG_FEATS}[FEATURE_SET]
print(FEATURE_SET, "->", len(SPEC_FEATS), "spectrogram +", len(EEG_FEATS), "EEG =", len(FEATS), "features")

spec -> 448 spectrogram + 0 EEG = 448 features


## 3. Soft-label training and patient-grouped cross-validation

**Soft labels in scikit-learn.** scikit-learn classifiers expect one class per sample. We therefore repeat each EEG
six times, once per class, and weight each copy by that class's vote fraction. Minimising the weighted
cross-entropy is then the same as minimising the KL divergence to the vote distribution.

**Cross-validation (CV).** The training data are split into 5 folds *by patient*. Each model is trained on 4 folds
and evaluated on the held-out fold; the out-of-fold predictions give an estimate of performance on unseen patients.
Grouping by patient matters: EEGs of the same patient are very similar, and a random split would overestimate
performance. We report the CV score on all EEGs and on EEGs with at least 10 votes (more reliable labels),
next to the score of a trivial model that always predicts the average vote distribution (the *prior*).

In [7]:
def kl_score(y_true, y_pred, eps=1e-15):
    """Mean KL divergence KL(y_true || y_pred) over samples, as in the competition metric."""
    p = np.clip(y_pred, eps, 1)
    p = p / p.sum(axis=1, keepdims=True)
    t = y_true
    return np.mean(np.sum(np.where(t > 0, t * np.log(np.where(t > 0, t, 1) / p), 0), axis=1))


def expand(X, Y):
    """Turn soft labels into weighted hard labels.

    Each sample is repeated once per class (label = class index, weight = vote fraction);
    copies with zero weight are dropped.
    """
    n, k = Y.shape
    Xr = np.repeat(X, k, axis=0)
    yr = np.tile(np.arange(k), n)
    wr = Y.reshape(-1)
    keep = wr > 0
    return Xr[keep], yr[keep], wr[keep]


# The two candidate models. Logistic regression needs standardised features; gradient boosting does not.
MODELS = {
    "logreg": lambda: make_pipeline(StandardScaler(), LogisticRegression(C=0.1, max_iter=2000)),
    "hgb": lambda: HistGradientBoostingClassifier(learning_rate=0.05, max_iter=300,
                                                  max_leaf_nodes=31, l2_regularization=1.0,
                                                  random_state=SEED),
}


def fit(name, X, Y):
    """Fit model `name` on features X and soft labels Y (rows are vote distributions)."""
    Xe, ye, we = expand(X, Y)
    m = MODELS[name]()
    if name == "logreg":   # in a Pipeline the weight must be routed to the final step
        m.fit(Xe, ye, logisticregression__sample_weight=we)
    else:
        m.fit(Xe, ye, sample_weight=we)
    return m


X = np.nan_to_num(df[FEATS].to_numpy(dtype=np.float32), nan=0.0, posinf=0.0, neginf=0.0)
Y = df[CLASSES].to_numpy()
groups = df["patient_id"].to_numpy()
prior = np.tile(Y.mean(axis=0), (len(Y), 1))   # trivial reference: always predict the mean distribution
hi = df["num_votes"].to_numpy() >= 10          # EEGs with at least 10 expert votes

if RUN_CV:
    oof = {name: np.zeros_like(Y) for name in MODELS}   # out-of-fold predictions per model
    for fold, (tr, va) in enumerate(GroupKFold(n_splits=N_FOLDS).split(X, Y, groups)):
        for name in MODELS:
            oof[name][va] = fit(name, X[tr], Y[tr]).predict_proba(X[va])
        print(f"fold {fold}: " + ", ".join(f"{n}={kl_score(Y[va], oof[n][va]):.4f}" for n in MODELS))
    print(f"\nCV KL (all EEGs)        prior={kl_score(Y, prior):.4f} " +
          " ".join(f"{n}={kl_score(Y, oof[n]):.4f}" for n in MODELS))
    print(f"CV KL (>=10 votes only) prior={kl_score(Y[hi], prior[hi]):.4f} " +
          " ".join(f"{n}={kl_score(Y[hi], oof[n][hi]):.4f}" for n in MODELS))
    BEST = min(MODELS, key=lambda n: kl_score(Y, oof[n]))   # lower KL is better
else:
    BEST = "hgb"   # default when CV is skipped
print("model for submission:", BEST)

model for submission: hgb


We voegen nog twee extra modellen bij en gaan na of dit verbetering brengt. Waarom juist deze? write up
opmerking: De regressors (ExtraTrees, MLP) hebben geen predict_proba en geen 6×-herhaling nodig.

In [8]:
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.neural_network import MLPRegressor


class SoftRegressor:
    """Regressor op de vote-fracties, met predict_proba die een kansverdeling teruggeeft."""                           #ExtraTrees en MLP hebben nu zelfde predict_proba-interface  als classifiers.
    def __init__(self, model):
        self.model = model

    def fit(self, X, Y):
        self.model.fit(X, Y)                      # multi-output: Y heeft 6 kolommen
        return self

    def predict_proba(self, X):
        p = np.clip(self.model.predict(X), 1e-6, None)
        return p / p.sum(axis=1, keepdims=True)


MODELS.update({
    "et":  lambda: ExtraTreesRegressor(n_estimators=300, min_samples_leaf=5, max_features=0.3,                     #voegt drie modellen toe aan bestaande dict. logreg en hgb blijven ongewijzigd.
                                       n_jobs=-1, random_state=SEED),
    "mlp": lambda: make_pipeline(StandardScaler(),
                                 MLPRegressor(hidden_layer_sizes=(128, 64), alpha=1e-2, early_stopping=True,
                                              random_state=SEED)),
    "hgb_small": lambda: HistGradientBoostingClassifier(learning_rate=0.03, max_iter=300,
                                                        max_leaf_nodes=15, min_samples_leaf=100,
                                                        l2_regularization=5.0, early_stopping=False,
                                                        random_state=SEED),
})
REGRESSORS = {"et", "mlp"}


def fit(name, X, Y):
    if name in REGRESSORS:
        return SoftRegressor(MODELS[name]()).fit(X, Y)
    Xe, ye, we = expand(X, Y)
    m = MODELS[name]()
    if name == "logreg":
        m.fit(Xe, ye, logisticregression__sample_weight=we)
    else:
        m.fit(Xe, ye, sample_weight=we)
    return m

Volgende code beslist welk model beste is

In [9]:
BEST = "et"      # gekozen op basis van de quick-CV (KL 1.0069 vs 1.0229 voor hgb_small)
print("model voor submission:", BEST)


model voor submission: et


In [10]:
oof_et = np.zeros_like(Y)
for tr, va in GroupKFold(n_splits=N_FOLDS).split(X, Y, groups):
    oof_et[va] = fit("et", X[tr], Y[tr]).predict_proba(X[va])
print(f"{FEATURE_SET}: {X.shape[1]} features | KL all {kl_score(Y, oof_et):.4f} | "
      f">=10 votes {kl_score(Y[hi], oof_et[hi]):.4f}")

spec: 448 features | KL all 0.9247 | >=10 votes 0.8720


## 4. Fit the chosen model on all training data and predict the test set

The visible `test.csv` contains a single example. When the notebook is submitted, Kaggle re-runs it
with the hidden test set in its place, so this cell must work for any number of test EEGs.

In [11]:
final = fit(BEST, X, Y)

test = pd.read_csv(f"{DATA}/test.csv")
T_df = build_features(test, f"{DATA}/test_spectrograms", is_test=True)
T_df = test[["eeg_id"]].merge(T_df, on="eeg_id", how="left")   # keep the order of test.csv
if FEATURE_SET != "spec":
    TE_df = build_eeg_features(test, f"{DATA}/test_eegs", is_test=True)
    T_df = T_df.merge(TE_df, on="eeg_id", how="left")
XT = np.nan_to_num(T_df[FEATS].to_numpy(dtype=np.float32), nan=0.0, posinf=0.0, neginf=0.0)

# Submission format: eeg_id plus one predicted probability per class (each row sums to 1).
sub = pd.DataFrame({"eeg_id": T_df["eeg_id"]})
sub[CLASSES] = final.predict_proba(XT)
sub.to_csv("submission.csv", index=False)
print(sub.shape)
sub.head()

1 spectrograms -> 1 feature rows in 0s
(1, 7)


,eeg_id,seizure_vote,lpd_vote,gpd_vote,lrda_vote,grda_vote,other_vote
0,3911565283,0.231097,0.101268,0.018689,0.159339,0.092183,0.397424


Beelden maken uit spec


In [12]:
import torch, torch.nn as nn, torch.nn.functional as F

CNN_ROWS = 100            # centrale 100 rijen = 200 s (1 rij = 2 s); mijn keuze, zelf te variëren
DEV = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEV)


def load_imgs(path, windows, is_test):
    """Per EEG een beeld (4 regio's x 100 frequenties x CNN_ROWS tijdstappen) uit één spectrogram-bestand."""
    spec = pd.read_parquet(path)
    cols = list(spec.columns[1:])
    arr = spec.iloc[:, 1:].to_numpy(dtype=np.float32)
    idx = [[i for i, c in enumerate(cols) if c.startswith(r + "_")] for r in REGIONS]
    out = []
    for eeg_id, off_min, off_max in windows:
        start = max(0, (len(arr) - 300) // 2) if is_test else int((off_min + off_max) // 4)   # zelfde als de baseline
        lo = start + 150 - CNN_ROWS // 2                                                       # midden van het 300-rijen venster
        seg = arr[lo: lo + CNN_ROWS]
        if len(seg) < CNN_ROWS:                                                                # vangnet: te kort spectrogram
            seg = np.pad(seg, ((0, CNN_ROWS - len(seg)), (0, 0)))
        img = np.stack([seg[:, ix].T for ix in idx])                                          # (4, 100, CNN_ROWS)
        img = np.log(np.clip(np.nan_to_num(img, nan=0.0), np.exp(-4), np.exp(8)))             # log-schaal, pieken begrensd
        img = (img - img.mean((1, 2), keepdims=True)) / (img.std((1, 2), keepdims=True) + 1e-6)
        out.append((eeg_id, img.astype(np.float16)))
    return out


def build_imgs(meta, spec_dir, is_test=False):
    t0 = time.time()
    if is_test:
        meta = meta.assign(off_min=0.0, off_max=0.0)
    jobs = [delayed(load_imgs)(f"{spec_dir}/{sid}.parquet", list(zip(g.eeg_id, g.off_min, g.off_max)), is_test)
            for sid, g in meta.groupby("spectrogram_id")]
    res = Parallel(n_jobs=N_JOBS, batch_size=32)(jobs)
    d = {e: im for rows in res for e, im in rows}
    print(f"{len(d)} beelden in {time.time() - t0:.0f}s")
    return d


imgs = build_imgs(df, f"{DATA}/train_spectrograms")
IMG = np.stack([imgs[e] for e in df["eeg_id"]])      # zelfde volgorde als df, dus als Y en groups
del imgs
print(IMG.shape, IMG.dtype)

device: cuda
4738 beelden in 30s
(4738, 4, 100, 100) float16


In [13]:
torch.manual_seed(SEED); np.random.seed(SEED)


class SmallCNN(nn.Module):
    def __init__(self):
        super().__init__()
        def blk(i, o):
            return nn.Sequential(nn.Conv2d(i, o, 3, padding=1), nn.BatchNorm2d(o), nn.ReLU(), nn.MaxPool2d(2))
        self.f = nn.Sequential(blk(4, 16), blk(16, 32), blk(32, 64), blk(64, 64))
        self.head = nn.Sequential(nn.Dropout(0.3), nn.Linear(64, 6))

    def forward(self, x):
        return self.head(self.f(x).mean((2, 3)))          # global average pooling, dan 6 logits


def train_cnn(Xtr, Ytr, epochs=8, bs=64, lr=1e-3):
    m = SmallCNN().to(DEV)
    opt = torch.optim.AdamW(m.parameters(), lr=lr, weight_decay=1e-2)
    Xt, Yt = torch.from_numpy(Xtr), torch.from_numpy(Ytr.astype(np.float32))
    for ep in range(epochs):
        m.train()
        perm = torch.randperm(len(Xt))
        for i in range(0, len(perm), bs):
            b = perm[i:i + bs]
            xb, yb = Xt[b].float().to(DEV), Yt[b].to(DEV)
            loss = F.kl_div(F.log_softmax(m(xb), 1), yb, reduction="batchmean")     # KL op de soft labels
            opt.zero_grad(); loss.backward(); opt.step()
    return m


@torch.no_grad()
def predict_cnn(m, X, bs=256):
    m.eval()
    return np.concatenate([F.softmax(m(torch.from_numpy(X[i:i + bs]).float().to(DEV)), 1).cpu().numpy()
                           for i in range(0, len(X), bs)])


oof_cnn = np.zeros_like(Y)
for k, (tr, va) in enumerate(GroupKFold(n_splits=N_FOLDS).split(IMG, Y, groups)):
    t0 = time.time()
    m = train_cnn(IMG[tr], Y[tr])
    oof_cnn[va] = predict_cnn(m, IMG[va])
    print(f"fold {k}: KL {kl_score(Y[va], oof_cnn[va]):.4f}  ({time.time() - t0:.0f}s)", flush=True)

print(f"CNN | KL all {kl_score(Y, oof_cnn):.4f} | >=10 votes {kl_score(Y[hi], oof_cnn[hi]):.4f}")

fold 0: KL 1.1312  (18s)
fold 1: KL 1.2542  (8s)
fold 2: KL 1.1381  (8s)
fold 3: KL 0.9933  (8s)
fold 4: KL 0.8931  (8s)
CNN | KL all 1.0821 | >=10 votes 0.9931
